# Eval config quicklook

Quick statistics and maps for a `crai-evaluate` run, driven entirely by the **eval config file**
(the same `--load-from-file` argument file used for the evaluation run).

The notebook resolves the output paths exactly like `crai-evaluate` does (via
`climatereconstructionai.config.set_evaluate_args`, including the
`{evaluation-dirs}/{config-name}/{run-name}/data/` layout), then computes:

- gridwise **RMSE maps** (output vs gt, infilled vs gt),
- gridwise **correlation map** and **bias map** (output vs gt),
- an RMSE **time series**,
- a scalar stats table (RMSE / MAE / bias / correlation) overall and split into
  the valid (`mask==1`) and hole (`mask==0`) regions when a mask file exists.

Run `crai-evaluate --load-from-file <your eval config>` first, then point `EVAL_CONFIG`
below at that config and run all cells.


In [1]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
EVAL_CONFIG = os.path.join(REPO_ROOT, "tests", "in", "evaluation", "minimum-1.inp")
RUN_CWD = REPO_ROOT
MEMBER = 0


## 1. Parse the eval config

`RUN_CWD` must be the directory from which `crai-evaluate` was started, because relative
paths inside the config file (`--evaluation-dirs`, `--model-dir`, ...) are resolved against it.
`EVAL_CONFIG` is the argument file of the evaluation run, e.g.
`input/levante/eval-cnn-cerra-t2m.txt`.


In [2]:
from climatereconstructionai import config as cfg
EVAL_CONFIG = "/work/uo1075/u244085/trees_in_the_city/ML_downscaling/Physics-Informed-CNN-Reconstructions/input/levante/eval-cnn-cerra-t2m-all_var-ger-2deg-el5-po0-cf128-l7-bilin-b64-b0cee0c0.txt"
EVAL_CONFIG = os.path.abspath(EVAL_CONFIG)
os.chdir(RUN_CWD)
cfg.set_evaluate_args(EVAL_CONFIG)

eval_dir = cfg.evaluation_dirs[0]
data_dir = os.path.join(eval_dir, "data")
images_dir = os.path.join(eval_dir, "images")
print("eval config:    ", EVAL_CONFIG)
print("evaluation dir: ", os.path.abspath(eval_dir))
print("eval names:     ", cfg.eval_names)
print("data types:     ", cfg.data_types)


eval config:     /work/uo1075/u244085/trees_in_the_city/ML_downscaling/Physics-Informed-CNN-Reconstructions/input/levante/eval-cnn-cerra-t2m-all_var-ger-2deg-el5-po0-cf128-l7-bilin-b64-b0cee0c0.txt
evaluation dir:  /work/uo1075/u244085/trees_in_the_city/ML_downscaling/Physics-Informed-CNN-Reconstructions/evaluations/t2m/cerra-t2m-all_var-ger-2deg-el5-po0-cf128-l7-bilin-b64-b0cee0c0
eval names:      ['output']
data types:      ['2t', '2r', 'sp', '10si', '10wdir', 'orog', '2t']


## 2. Locate the evaluation output files

Merged files (`{eval_name}_{gt,mask,image,output,infilled}.nc`) are used when present;
with `--split-outputs` the per-split files are collected instead.


In [3]:
CNAMES = ["gt", "mask", "image", "output", "infilled"]


def find_outputs():
    found = {}
    for name in cfg.eval_names:
        for cname in CNAMES:
            merged = os.path.join(data_dir, f"{name}_{cname}.nc")
            if os.path.isfile(merged):
                found[(name, cname)] = merged
                continue
            parts = sorted(glob.glob(os.path.join(data_dir, f"{name}_{cname}.*-*.nc")))
            if parts:
                found[(name, cname)] = parts
    return found


outputs = find_outputs()
for (name, cname), path in sorted(outputs.items()):
    desc = path if isinstance(path, str) else f"{len(path)} split files"
    print(f"{name}_{cname}: {desc}")
if not outputs:
    raise FileNotFoundError(
        f"No evaluation outputs found in {data_dir}. "
        f"Run: crai-evaluate --load-from-file {EVAL_CONFIG}"
    )


FileNotFoundError: No evaluation outputs found in ./evaluations//t2m/cerra-t2m-all_var-ger-2deg-el5-po0-cf128-l7-bilin-b64-b0cee0c0/data. Run: crai-evaluate --load-from-file /work/uo1075/u244085/trees_in_the_city/ML_downscaling/Physics-Informed-CNN-Reconstructions/input/levante/eval-cnn-cerra-t2m-all_var-ger-2deg-el5-po0-cf128-l7-bilin-b64-b0cee0c0.txt

## 3. Statistics helpers


In [4]:
def open_out(entry):
    if isinstance(entry, list):
        return xr.open_mfdataset(entry, combine="nested", concat_dim="time", chunks={})
    return xr.open_dataset(entry)


def data_var(ds):
    return list(ds.data_vars)[0]


def prep(ds):
    da = ds[data_var(ds)]
    for d in ("member", "depth", "height"):
        if d in da.dims:
            da = da.isel({d: MEMBER if d == "member" else 0})
    return da


def time_dim(da):
    return "time" if "time" in da.dims else da.dims[0]


def spatial_dims(da):
    return [d for d in da.dims if d != time_dim(da)]


def gridwise_rmse(a, b):
    return np.sqrt(((a - b) ** 2).mean(dim=time_dim(a), skipna=True))


def gridwise_corr(a, b):
    t = time_dim(a)
    a0 = a - a.mean(dim=t, skipna=True)
    b0 = b - b.mean(dim=t, skipna=True)
    return (a0 * b0).sum(dim=t) / np.sqrt((a0 ** 2).sum(dim=t) * (b0 ** 2).sum(dim=t))


def scalar_stats(pred, ref, use=None):
    p = np.asarray(pred.values).ravel()
    r = np.asarray(ref.values).ravel()
    ok = ~(np.isnan(p) | np.isnan(r))
    if use is not None:
        ok &= np.broadcast_to(np.asarray(use), pred.shape).ravel()
    p, r = p[ok], r[ok]
    if p.size == 0:
        return {"n": 0, "RMSE": np.nan, "MAE": np.nan, "bias": np.nan, "corr": np.nan}
    corr = np.corrcoef(p, r)[0, 1] if np.std(p) > 0 and np.std(r) > 0 else np.nan
    return {
        "n": int(p.size),
        "RMSE": float(np.sqrt(np.mean((p - r) ** 2))),
        "MAE": float(np.mean(np.abs(p - r))),
        "bias": float(np.mean(p - r)),
        "corr": float(corr),
    }


## 4. Compute stats per eval name

If the evaluated mask is fully valid (all ones), `infilled == gt` and there is no hole
region, so the run measures pure reconstruction of the observed field.


In [5]:
summary_rows = []
maps = {}
for name in cfg.eval_names:
    paths = {c: outputs[(name, c)] for c in CNAMES if (name, c) in outputs}
    if "gt" not in paths or "output" not in paths:
        print(f"skipping '{name}': need at least gt and output files")
        continue

    gt = prep(open_out(paths["gt"]))
    out = prep(open_out(paths["output"]))
    inf = prep(open_out(paths["infilled"])) if "infilled" in paths else None
    mask = prep(open_out(paths["mask"])) if "mask" in paths else None

    t = time_dim(gt)
    sp = spatial_dims(gt)
    maps[name] = {
        "rmse_output": gridwise_rmse(out, gt),
        "corr": gridwise_corr(out, gt),
        "bias": (out - gt).mean(dim=t, skipna=True),
        "rmse_ts": np.sqrt(((out - gt) ** 2).mean(dim=sp, skipna=True)),
        "rmse_infilled": gridwise_rmse(inf, gt) if inf is not None else None,
        "rmse_inf_ts": np.sqrt(((inf - gt) ** 2).mean(dim=sp, skipna=True)) if inf is not None else None,
    }

    valid = mask == 1 if mask is not None else None
    hole = mask == 0 if (mask is not None and bool((mask == 0).any())) else None

    for label, pred in [("output", out), ("infilled", inf)]:
        if pred is None:
            continue
        summary_rows.append({"eval": name, "field": label, "region": "all", **scalar_stats(pred, gt)})
        if valid is not None:
            summary_rows.append({"eval": name, "field": label, "region": "valid", **scalar_stats(pred, gt, use=valid)})
        if hole is not None:
            summary_rows.append({"eval": name, "field": label, "region": "hole", **scalar_stats(pred, gt, use=hole)})

summary = pd.DataFrame(summary_rows)
summary


skipping 'output': need at least gt and output files


""


## 5. Maps and time series

Figures are also saved as `quicklook_*.png` into the `images/` folder of the evaluation dir.


In [6]:
def lat_lon(da):
    lat = next((c for c in ("lat", "latitude", "y") if c in da.dims or c in da.coords), None)
    lon = next((c for c in ("lon", "longitude", "x") if c in da.dims or c in da.coords), None)
    return lat, lon


def plot_map(da, title, cmap="YlOrRd", vmin=None, vmax=None):
    lat, lon = lat_lon(da)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    if lat is not None and lon is not None:
        mesh = ax.pcolormesh(da[lon].values, da[lat].values, np.asarray(da),
                             cmap=cmap, vmin=vmin, vmax=vmax, shading="auto")
        ax.set_xlabel(lon)
        ax.set_ylabel(lat)
    else:
        mesh = ax.imshow(np.asarray(da), origin="lower", cmap=cmap, vmin=vmin, vmax=vmax)
    fig.colorbar(mesh, ax=ax)
    ax.set_title(title)
    fig.tight_layout()
    return fig


os.makedirs(images_dir, exist_ok=True)
for name, m in maps.items():
    figs = [
        (plot_map(m["rmse_output"], f"{name}: RMSE map (output vs gt)", vmin=0), f"quicklook_{name}_rmse_output.png"),
        (plot_map(m["corr"], f"{name}: correlation map (output vs gt)", cmap="RdBu_r", vmin=-1, vmax=1), f"quicklook_{name}_corr.png"),
        (plot_map(m["bias"], f"{name}: bias map (output - gt)", cmap="RdBu_r"), f"quicklook_{name}_bias.png"),
    ]
    if m["rmse_infilled"] is not None:
        figs.append((plot_map(m["rmse_infilled"], f"{name}: RMSE map (infilled vs gt)", vmin=0), f"quicklook_{name}_rmse_infilled.png"))
    for fig, fname in figs:
        fig.savefig(os.path.join(images_dir, fname), dpi=150)
        plt.show()

    t = time_dim(m["rmse_ts"])
    tvals = m["rmse_ts"].coords[t].values if t in m["rmse_ts"].coords else np.arange(m["rmse_ts"].size)
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(tvals, np.asarray(m["rmse_ts"]), label="output")
    if m["rmse_inf_ts"] is not None:
        ax.plot(tvals, np.asarray(m["rmse_inf_ts"]), label="infilled")
    ax.set_title(f"{name}: RMSE time series")
    ax.set_ylabel("RMSE")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(images_dir, f"quicklook_{name}_rmse_ts.png"), dpi=150)
    plt.show()

print("figures saved to:", os.path.abspath(images_dir))


figures saved to: /work/uo1075/u244085/trees_in_the_city/ML_downscaling/Physics-Informed-CNN-Reconstructions/evaluations/t2m/cerra-t2m-all_var-ger-2deg-el5-po0-cf128-l7-bilin-b64-b0cee0c0/images


## Notes

- Point `EVAL_CONFIG` at any eval argument file (e.g. `input/levante/eval-cnn-cerra-t2m.txt`)
  and set `RUN_CWD` to the directory you started `crai-evaluate` from, so relative paths in the
  config resolve the same way as during the run.
- If several models were evaluated, they are stacked on the `member` dimension of the merged
  files; `MEMBER` selects which one is shown.
- With `--split-outputs`, the per-split files (`{name}_{cname}.{model}-{split}.nc`) are merged
  on the fly along `time`.
- If the mask is fully valid, `infilled` equals `gt` and the hole-region rows are omitted.
